In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

movies = pd.read_csv('data/u.item', sep='|', encoding='latin-1', header=None, usecols=[0, 1], names=['item_id', 'title'])
ratings = pd.read_csv('data/u.data', sep='\t', header=None, names=['user_id', 'item_id', 'rating', 'timestamp'])

df = pd.merge(ratings, movies, on='item_id')
print(f"Dataset shape: {df.shape}")
print(f"Utilisateurs uniques: {df['user_id'].nunique()}")
print(f"Films uniques: {df['item_id'].nunique()}")

Dataset shape: (100000, 5)
Utilisateurs uniques: 943
Films uniques: 1682


In [2]:
# Création de la matrice (lignes=utilisateurs, colonnes=films, valeurs=notes)
user_item_matrix = df.pivot_table(index='user_id', columns='item_id', values='rating')

print(f"Taille de la matrice: {user_item_matrix.shape}")
print(f"Densité de la matrice: {100 * (1 - user_item_matrix.isnull().sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1])):.2f}%")

Taille de la matrice: (943, 1682)
Densité de la matrice: 6.30%


In [3]:
# On remplit les NaN par 0 pour le calcul de similarité
matrix_filled = user_item_matrix.fillna(0)

# Calcul de la similarité entre les films (colonnes)
item_similarity = cosine_similarity(matrix_filled.T)
item_sim_df = pd.DataFrame(item_similarity, index=matrix_filled.columns, columns=matrix_filled.columns)

def get_similar_movies(movie_title, top_n=5):
    movie_id = movies[movies['title'] == movie_title]['item_id'].values[0]
    similar_scores = item_sim_df[movie_id].sort_values(ascending=False)
    similar_movies = movies[movies['item_id'].isin(similar_scores.index)]
    similar_movies['similarity'] = similar_scores.values
    return similar_movies.sort_values('similarity', ascending=False).head(top_n)

print("Films similaires à 'Star Wars (1977)':")
display(get_similar_movies('Star Wars (1977)'))

Films similaires à 'Star Wars (1977)':


,item_id,title,similarity
0,1,Toy Story (1995),1.000000
1,2,GoldenEye (1995),0.884476
2,3,Four Rooms (1995),0.764885
3,4,Get Shorty (1995),0.749819
4,5,Copycat (1995),0.734572


In [5]:
# Préparation pour l'évaluation (Train/Test split sur les notes)
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

# Création des matrices d'entraînement
train_matrix = train_data.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)

# Application de TruncatedSVD (réduction de dimension pour capturer les "goûts" latents)
n_factors = 20
svd = TruncatedSVD(n_components=n_factors, random_state=42)
matrix_factors = svd.fit_transform(train_matrix)

# Reconstruction de la matrice pour faire des prédictions
predicted_ratings = np.dot(matrix_factors, svd.components_)
predicted_df = pd.DataFrame(predicted_ratings, index=train_matrix.index, columns=train_matrix.columns)

# Évaluation simplifiée et robuste
predictions = []
actuals = []

for _, row in test_data.iterrows():
    user = row['user_id']
    item = row['item_id']
    actual_rating = row['rating']
    
    # Si l'utilisateur et le film sont dans notre matrice de prédiction
    if user in predicted_df.index and item in predicted_df.columns:
        pred = predicted_df.loc[user, item]
        predictions.append(pred)
        actuals.append(actual_rating)

rmse = np.sqrt(mean_squared_error(actuals, predictions))
print(f"RMSE du modèle SVD sur le set de test: {rmse:.4f}")
print(f"Nombre de prédictions évaluées: {len(predictions)}")

RMSE du modèle SVD sur le set de test: 2.6431
Nombre de prédictions évaluées: 19971


In [6]:
def recommend_for_user(user_id, top_n=5):
    # 1. Récupérer les films déjà vus par l'utilisateur
    user_ratings = df[df['user_id'] == user_id]
    seen_movies = user_ratings['item_id'].tolist()
    
    # 2. Récupérer les prédictions pour cet utilisateur
    if user_id in predicted_df.index:
        # On garde seulement les films non vus
        user_predictions = predicted_df.loc[user_id].drop(seen_movies, errors='ignore')
        
        # On trie par note prédite décroissante
        top_recommendations = user_predictions.sort_values(ascending=False).head(top_n)
        
        # 3. On joint avec les titres des films pour l'affichage
        rec_movies = movies[movies['item_id'].isin(top_recommendations.index)]
        rec_movies['predicted_rating'] = top_recommendations.values
        
        return rec_movies.sort_values('predicted_rating', ascending=False).reset_index(drop=True)
    else:
        return f"L'utilisateur {user_id} n'a pas été trouvé dans le dataset d'entraînement."

print("🎬 Recommandations personnalisées pour l'utilisateur 10 :")
display(recommend_for_user(10))

🎬 Recommandations personnalisées pour l'utilisateur 10 :


,item_id,title,predicted_rating
0,187,"Godfather: Part II, The (1974)",3.391095
1,318,Schindler's List (1993),3.376384
2,427,To Kill a Mockingbird (1962),2.843932
3,514,Annie Hall (1977),2.795043
4,515,"Boot, Das (1981)",2.672757


In [7]:
import mlflow
import mlflow.sklearn

# Définir le nom de l'expérience
mlflow.set_experiment("Recommandation_System_MovieLens")

with mlflow.start_run(run_name="SVD_Model_Baseline"):
    # Log des hyperparamètres
    mlflow.log_param("n_factors", n_factors)
    mlflow.log_param("model_type", "TruncatedSVD")
    mlflow.log_param("test_size", 0.2)
    
    # Log des métriques
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("Explained_Variance_Ratio", float(svd.explained_variance_ratio_.sum()))
    
    print("✅ MLflow run logged successfully.")
    print(f"📁 Les données sont sauvegardées dans le dossier 'mlruns' de ton projet.")

2026/11/10 00:02:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/10 00:02:01 INFO mlflow.store.db.utils: Updating database tables
2026/11/10 00:02:03 INFO mlflow.tracking.fluent: Experiment with name 'Recommandation_System_MovieLens' does not exist. Creating a new experiment.


✅ MLflow run logged successfully.
📁 Les données sont sauvegardées dans le dossier 'mlruns' de ton projet.
